# 04 - Regime Analysis (Investment Clock)

Detección de regímenes de mercado basado en el modelo **Investment Clock** con 4 fases:

| Fase | Características | Estrategias Favorecidas |
|------|----------------|------------------------|
| **Expansion** | Crecimiento + baja vol + momentum positivo | Growth, Momentum, Risk-on |
| **Slowdown** | Crecimiento desacelerando + vol subiendo | Value, Large Cap, Risk Parity |
| **Recession** | Contracción + alta volatilidad | Min Variance, Risk Parity |
| **Recovery** | Mejora desde mínimos + vol cayendo | Value, Small Cap, Equal Weight |

## Métodos de Detección
1. **Heuristic**: Reglas simples (vol + tendencia)
2. **Fuzzy Logic**: Lógica difusa con funciones de membresía
3. **HMM**: Hidden Markov Model multivariado
4. **Clustering**: GMM sobre features de mercado

## Prerequisitos
- Ejecutar `03_algos_analysis.ipynb` primero

## Output
- `data/processed/regime_labels.parquet`: régimen por día
- `configs/regime_cluster_config.yaml`: configuración combinada

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.analysis.regime_detector import (
    RegimeDetector, RegimeMethod,
    REGIME_EXPANSION, REGIME_SLOWDOWN, REGIME_RECESSION, REGIME_RECOVERY,
    INVESTMENT_CLOCK_REGIMES
)
from src.utils.logging_utils import setup_logging

setup_logging(level='INFO')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Cargar Datos

In [ ]:
# Cargar datos del benchmark
loader = DataLoader('../data/raw/')
preprocessor = DataPreprocessor(initial_capital=100, resample_freq='D')

# Benchmark
benchmark_raw = loader.load_benchmark()
benchmark = preprocessor.process_benchmark(benchmark_raw)

# Algoritmos del benchmark para calcular returns
benchmark_algo_ids = set(benchmark.products)
algorithms_benchmark = loader.load_all_algorithms(algo_ids=list(benchmark_algo_ids), show_progress=True)
processed_benchmark_algos = preprocessor.process_all_algorithms(algorithms_benchmark, show_progress=True)
returns_matrix_benchmark = preprocessor.create_returns_matrix(processed_benchmark_algos)

# Calcular retornos diarios del benchmark
benchmark_daily_returns = preprocessor.calculate_benchmark_daily_returns(
    returns_matrix_benchmark, benchmark.weights
)

print(f"\n{'='*60}")
print(f"DATOS DEL BENCHMARK")
print(f"{'='*60}")
print(f"Algoritmos en benchmark: {len(benchmark_algo_ids)}")
print(f"Rango fechas: {benchmark_daily_returns.index.min()} -> {benchmark_daily_returns.index.max()}")
print(f"Días de trading: {len(benchmark_daily_returns)}")

In [ ]:
# Cargar clusters de algoritmos (generados en 03_algos_analysis)
try:
    algo_features = pd.read_parquet('../data/processed/algo_features_enhanced.parquet')
    algo_clusters = pd.read_parquet('../data/processed/algo_clusters_twolayer.parquet')
    print("Cargados datos de two-layer clustering")
    print(f"  - Features: {len(algo_features)} algoritmos")
    print(f"  - Life clusters: {algo_clusters['life_cluster_name'].nunique()}")
    cluster_col = 'life_cluster_name'
    clusters_available = True
except FileNotFoundError:
    try:
        algo_clusters = pd.read_parquet('../data/processed/algo_clusters.parquet')
        print("Cargados datos de clustering simple")
        cluster_col = 'cluster_name'
        clusters_available = True
        algo_features = None
    except FileNotFoundError:
        print("ADVERTENCIA: No se encontraron datos de clustering")
        print("Ejecuta primero 03_algos_analysis.ipynb")
        clusters_available = False

---
# PARTE 1: Comparación de Métodos de Detección
---

Comparamos los 4 métodos disponibles para detectar las 4 fases del Investment Clock.

In [ ]:
# Detectar regímenes con cada método
print("="*80)
print("DETECCIÓN DE REGÍMENES - COMPARACIÓN DE MÉTODOS")
print("="*80)

methods = {
    'Heuristic': RegimeMethod.HEURISTIC,
    'Fuzzy Logic': RegimeMethod.FUZZY,
    'HMM': RegimeMethod.HMM,
    'Clustering (GMM)': RegimeMethod.CLUSTERING,
}

all_regimes = {}
all_stats = {}

for name, method in methods.items():
    print(f"\n--- {name} ---")
    
    detector = RegimeDetector(
        method=method,
        vol_window=21,
        trend_window=63,
        momentum_window=21,
        smoothing_window=5,
    )
    
    regimes = detector.detect(benchmark_daily_returns)
    all_regimes[name] = regimes
    
    # Estadísticas
    stats = detector.get_regime_statistics(benchmark_daily_returns, regimes)
    all_stats[name] = stats
    
    print(f"Distribución:")
    for regime in INVESTMENT_CLOCK_REGIMES:
        count = (regimes == regime).sum()
        pct = count / len(regimes) * 100
        print(f"  {regime:12s}: {count:4d} días ({pct:5.1f}%)")

In [ ]:
# Mostrar estadísticas por método
print("\n" + "="*80)
print("ESTADÍSTICAS POR RÉGIMEN - CADA MÉTODO")
print("="*80)

for name, stats in all_stats.items():
    print(f"\n--- {name} ---")
    if len(stats) > 0:
        display(stats.style.format({
            'n_days': '{:.0f}',
            'pct_time': '{:.1f}%',
            'ann_return': '{:.2%}',
            'ann_vol': '{:.2%}',
            'sharpe': '{:.2f}',
            'max_dd': '{:.2%}',
            'avg_duration_days': '{:.1f}'
        }))
    else:
        print("  No hay suficientes datos")

In [ ]:
# Visualizar regímenes de cada método
fig, axes = plt.subplots(len(methods), 1, figsize=(14, 4*len(methods)), sharex=True)

equity = (1 + benchmark_daily_returns).cumprod() * 100

regime_colors = {
    REGIME_EXPANSION: 'lightgreen',
    REGIME_SLOWDOWN: 'khaki',
    REGIME_RECESSION: 'lightcoral',
    REGIME_RECOVERY: 'lightblue',
    'unknown': 'lightgray',
}

for idx, (name, regimes) in enumerate(all_regimes.items()):
    ax = axes[idx]
    ax.plot(equity.index, equity.values, 'k-', linewidth=1, alpha=0.7)
    
    for regime, color in regime_colors.items():
        mask = regimes == regime
        if mask.any():
            ax.fill_between(
                equity.index, equity.min(), equity.max(),
                where=mask.reindex(equity.index).fillna(False),
                alpha=0.4, color=color, label=regime
            )
    
    ax.set_ylabel('Equity')
    ax.set_title(f'{name}')
    if idx == 0:
        ax.legend(loc='upper left', ncol=4, fontsize=8)

plt.suptitle('Comparación de Métodos de Detección de Régimen', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Concordancia entre métodos
print("\n" + "="*60)
print("CONCORDANCIA ENTRE MÉTODOS")
print("="*60)

# Calcular % de acuerdo entre pares de métodos
method_names = list(all_regimes.keys())
concordance = pd.DataFrame(index=method_names, columns=method_names, dtype=float)

for m1 in method_names:
    for m2 in method_names:
        # Filtrar unknown
        mask = (all_regimes[m1] != 'unknown') & (all_regimes[m2] != 'unknown')
        if mask.sum() > 0:
            agreement = (all_regimes[m1][mask] == all_regimes[m2][mask]).mean()
            concordance.loc[m1, m2] = agreement * 100

print("\n% de días con mismo régimen:")
display(concordance.style.format('{:.1f}%').background_gradient(cmap='RdYlGn', vmin=30, vmax=100))

## 2. Selección del Método Óptimo

Evaluamos qué método produce regímenes más interpretables:
- Sharpe diferenciado entre regímenes (expansion > recession)
- Volatilidad coherente (recession > expansion)
- Duraciones razonables (no cambios muy frecuentes)

In [ ]:
# Evaluar calidad de cada método
print("="*80)
print("EVALUACIÓN DE CALIDAD DE MÉTODOS")
print("="*80)

quality_scores = []

for name, stats in all_stats.items():
    if len(stats) < 3:
        quality_scores.append({'method': name, 'score': 0, 'reason': 'Pocos regímenes detectados'})
        continue
    
    score = 0
    details = []
    
    # 1. Sharpe de expansion > sharpe de recession
    if REGIME_EXPANSION in stats.index and REGIME_RECESSION in stats.index:
        exp_sharpe = stats.loc[REGIME_EXPANSION, 'sharpe']
        rec_sharpe = stats.loc[REGIME_RECESSION, 'sharpe']
        if exp_sharpe > rec_sharpe:
            score += 25
            details.append(f"Sharpe OK: exp={exp_sharpe:.2f} > rec={rec_sharpe:.2f}")
        else:
            details.append(f"Sharpe FAIL: exp={exp_sharpe:.2f} <= rec={rec_sharpe:.2f}")
    
    # 2. Volatilidad de recession > volatilidad de expansion
    if REGIME_EXPANSION in stats.index and REGIME_RECESSION in stats.index:
        exp_vol = stats.loc[REGIME_EXPANSION, 'ann_vol']
        rec_vol = stats.loc[REGIME_RECESSION, 'ann_vol']
        if rec_vol > exp_vol:
            score += 25
            details.append(f"Vol OK: rec={rec_vol:.1%} > exp={exp_vol:.1%}")
        else:
            details.append(f"Vol FAIL: rec={rec_vol:.1%} <= exp={exp_vol:.1%}")
    
    # 3. Duración media razonable (>10 días)
    avg_duration = stats['avg_duration_days'].mean()
    if avg_duration >= 10:
        score += 25
        details.append(f"Duration OK: {avg_duration:.1f} días")
    else:
        details.append(f"Duration SHORT: {avg_duration:.1f} días")
    
    # 4. Distribución balanceada (ningún régimen <5% o >60%)
    pct_times = stats['pct_time']
    if pct_times.min() >= 5 and pct_times.max() <= 60:
        score += 25
        details.append(f"Balance OK: {pct_times.min():.1f}% - {pct_times.max():.1f}%")
    else:
        details.append(f"Balance FAIL: {pct_times.min():.1f}% - {pct_times.max():.1f}%")
    
    quality_scores.append({
        'method': name,
        'score': score,
        'details': '; '.join(details)
    })
    
    print(f"\n{name}: {score}/100")
    for d in details:
        print(f"  - {d}")

quality_df = pd.DataFrame(quality_scores).set_index('method').sort_values('score', ascending=False)
print("\n" + "="*60)
print("RANKING DE MÉTODOS:")
print("="*60)
display(quality_df)

In [ ]:
# Seleccionar el mejor método
best_method_name = quality_df['score'].idxmax()
selected_regimes = all_regimes[best_method_name]
selected_stats = all_stats[best_method_name]

print(f"\n{'='*60}")
print(f"MÉTODO SELECCIONADO: {best_method_name}")
print(f"{'='*60}")
print(f"Score: {quality_df.loc[best_method_name, 'score']}/100")

print("\nDistribución:")
for regime in INVESTMENT_CLOCK_REGIMES:
    count = (selected_regimes == regime).sum()
    pct = count / len(selected_regimes) * 100
    print(f"  {regime:12s}: {count:4d} días ({pct:5.1f}%)")

print("\nEstadísticas:")
display(selected_stats.style.format({
    'n_days': '{:.0f}',
    'pct_time': '{:.1f}%',
    'ann_return': '{:.2%}',
    'ann_vol': '{:.2%}',
    'sharpe': '{:.2f}',
    'max_dd': '{:.2%}',
    'avg_duration_days': '{:.1f}'
}))

In [ ]:
# Visualizar el método seleccionado en detalle
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Equity con regímenes
ax1 = axes[0]
ax1.plot(equity.index, equity.values, 'k-', linewidth=1.5, alpha=0.8)

for regime, color in regime_colors.items():
    mask = selected_regimes == regime
    if mask.any():
        ax1.fill_between(
            equity.index, equity.min() * 0.95, equity.max() * 1.05,
            where=mask.reindex(equity.index).fillna(False),
            alpha=0.4, color=color, label=regime.capitalize()
        )

ax1.set_ylabel('Equity', fontsize=12)
ax1.set_title(f'Regímenes Detectados ({best_method_name})', fontsize=14)
ax1.legend(loc='upper left', fontsize=10, ncol=4)

# Rolling volatility
ax2 = axes[1]
rolling_vol = benchmark_daily_returns.rolling(21).std() * np.sqrt(252)
ax2.plot(rolling_vol.index, rolling_vol.values, 'b-', linewidth=1)
ax2.axhline(rolling_vol.median(), color='red', linestyle='--', alpha=0.7, 
            label=f'Mediana: {rolling_vol.median():.1%}')
ax2.fill_between(rolling_vol.index, rolling_vol.values, alpha=0.3)
ax2.set_ylabel('Volatilidad (21d)', fontsize=12)
ax2.legend()

# Rolling trend
ax3 = axes[2]
rolling_trend = benchmark_daily_returns.rolling(63).mean() * 252
ax3.plot(rolling_trend.index, rolling_trend.values, 'g-', linewidth=1)
ax3.axhline(0, color='red', linestyle='--', alpha=0.7)
ax3.fill_between(rolling_trend.index, 0, rolling_trend.values, 
                 where=rolling_trend > 0, alpha=0.3, color='green', label='Positive')
ax3.fill_between(rolling_trend.index, 0, rolling_trend.values, 
                 where=rolling_trend <= 0, alpha=0.3, color='red', label='Negative')
ax3.set_ylabel('Trend (63d)', fontsize=12)
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de transición
print("\nMatriz de Transición:")
detector = RegimeDetector(method=methods[best_method_name])
transition_matrix = detector.get_transition_matrix(selected_regimes)
display(transition_matrix.style.format('{:.1%}').background_gradient(cmap='Blues'))

# Visualizar como heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(transition_matrix * 100, annot=True, fmt='.1f', cmap='Blues', 
            ax=ax, cbar_kws={'label': 'Probability (%)'})
ax.set_title('Matriz de Transición entre Regímenes')
ax.set_xlabel('Régimen Destino')
ax.set_ylabel('Régimen Origen')
plt.tight_layout()
plt.show()

print("\nInterpretación:")
print("  - Las transiciones deberían seguir el ciclo: expansion → slowdown → recession → recovery")
print("  - Alta persistencia en diagonal indica regímenes estables")

## 3. Guardar Regímenes Seleccionados

In [ ]:
# Guardar regímenes
selected_regimes.name = 'regime'
selected_regimes.to_frame().to_parquet('../data/processed/regime_labels.parquet')
print(f"Guardado: data/processed/regime_labels.parquet")
print(f"  Método: {best_method_name}")
print(f"  Regímenes: {', '.join(INVESTMENT_CLOCK_REGIMES)}")

---
# PARTE 2: Análisis Cruzado - Benchmark vs Regímenes
---

In [ ]:
# Cargar todos los algoritmos para análisis cruzado
if clusters_available:
    print("Cargando todos los algoritmos para análisis cruzado...")
    all_algo_ids = loader.list_algorithms()
    all_algorithms = loader.load_all_algorithms(algo_ids=all_algo_ids, show_progress=True)
    processed_all_algos = preprocessor.process_all_algorithms(all_algorithms, show_progress=True)
    returns_matrix_all = preprocessor.create_returns_matrix(processed_all_algos)
    print(f"Matriz de retornos: {returns_matrix_all.shape}")
else:
    print("Clusters no disponibles - saltando análisis cruzado")

In [ ]:
if clusters_available:
    # Pesos por cluster
    weights = benchmark.weights.copy()
    if hasattr(weights.index, 'tz') and weights.index.tz is not None:
        weights.index = weights.index.tz_localize(None)
    
    active_weights = weights[weights.sum(axis=1) > 0]
    
    # Mapping de clusters
    cluster_mapping = algo_clusters[cluster_col].to_dict()
    cluster_mapping = {k: v for k, v in cluster_mapping.items() if pd.notna(v)}
    
    def aggregate_weights_by_cluster(weights_row):
        cluster_weights = {}
        for algo_id, weight in weights_row.items():
            if algo_id in cluster_mapping:
                cluster = cluster_mapping[algo_id]
                cluster_weights[cluster] = cluster_weights.get(cluster, 0) + weight
        return pd.Series(cluster_weights)
    
    weights_by_cluster = active_weights.apply(aggregate_weights_by_cluster, axis=1).fillna(0)
    
    print(f"Pesos agregados por cluster: {weights_by_cluster.shape}")
    print(f"\nPeso medio por cluster:")
    display(weights_by_cluster.mean().sort_values(ascending=False).to_frame('avg_weight'))

In [ ]:
if clusters_available:
    # Asignación por cluster según régimen
    regimes_aligned = selected_regimes.reindex(weights_by_cluster.index)
    
    weights_regime = weights_by_cluster.copy()
    weights_regime['regime'] = regimes_aligned
    weights_regime = weights_regime[weights_regime['regime'] != 'unknown']
    
    cluster_cols = list(weights_by_cluster.columns)
    allocation_by_regime = weights_regime.groupby('regime')[cluster_cols].mean()
    
    # Reordenar por Investment Clock
    regime_order = [r for r in INVESTMENT_CLOCK_REGIMES if r in allocation_by_regime.index]
    allocation_by_regime = allocation_by_regime.reindex(regime_order)
    
    print("Peso medio por cluster en cada régimen:")
    display(allocation_by_regime.style.format('{:.2%}').background_gradient(cmap='YlOrRd', axis=1))

In [ ]:
if clusters_available:
    # Visualizar asignación por régimen
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Heatmap
    ax1 = axes[0]
    sns.heatmap(allocation_by_regime.T * 100, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax1)
    ax1.set_title('Asignación por Cluster según Régimen (%)')
    ax1.set_xlabel('Régimen')
    ax1.set_ylabel('Cluster')
    
    # Desviación vs media
    ax2 = axes[1]
    mean_alloc = allocation_by_regime.mean()
    deviation = (allocation_by_regime - mean_alloc) * 100
    sns.heatmap(deviation.T, annot=True, fmt='+.1f', cmap='RdYlGn', center=0, ax=ax2)
    ax2.set_title('Desviación vs Media (puntos porcentuales)')
    ax2.set_xlabel('Régimen')
    ax2.set_ylabel('Cluster')
    
    plt.tight_layout()
    plt.show()

## 4. Guardar Configuración

In [ ]:
import yaml
import os

# Configuración
config = {
    'regime_method': best_method_name.lower().replace(' ', '_'),
    'regime_names': INVESTMENT_CLOCK_REGIMES,
    'quality_score': int(quality_df.loc[best_method_name, 'score']),
}

if clusters_available:
    config['n_algo_clusters'] = len(set(cluster_mapping.values()))
    config['cluster_names'] = list(set(cluster_mapping.values()))
    config['allocation_by_regime'] = {
        k: {kk: float(vv) for kk, vv in v.items()}
        for k, v in allocation_by_regime.to_dict().items()
    }

os.makedirs('../configs', exist_ok=True)
with open('../configs/regime_cluster_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Guardado: configs/regime_cluster_config.yaml")

In [ ]:
# Resumen final
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║               RESUMEN: DETECCIÓN DE REGÍMENES (INVESTMENT CLOCK)         ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  MODELO: Investment Clock con 4 fases                                    ║
║  ─────────────────────────────────────                                   ║
║    ┌─────────────┐     ┌─────────────┐                                   ║
║    │  EXPANSION  │────▶│  SLOWDOWN   │                                   ║
║    │  (Risk-on)  │     │(Transición) │                                   ║
║    └──────▲──────┘     └──────┬──────┘                                   ║
║           │                   │                                          ║
║           │                   ▼                                          ║
║    ┌──────┴──────┐     ┌─────────────┐                                   ║
║    │  RECOVERY   │◀────│  RECESSION  │                                   ║
║    │(Transición) │     │ (Risk-off)  │                                   ║
║    └─────────────┘     └─────────────┘                                   ║
║                                                                          ║""")

print(f"║  MÉTODO SELECCIONADO: {best_method_name:<47}║")
print(f"║  Score de calidad: {quality_df.loc[best_method_name, 'score']}/100{' '*50}║")
print("║                                                                          ║")
print("║  DISTRIBUCIÓN DE REGÍMENES                                               ║")
print("║  ──────────────────────────                                              ║")

for regime in INVESTMENT_CLOCK_REGIMES:
    if regime in selected_stats.index:
        row = selected_stats.loc[regime]
        line = f"    {regime:12s}: {row['pct_time']:5.1f}% | Sharpe: {row['sharpe']:+.2f} | Vol: {row['ann_vol']:.1%}"
        print(f"║{line:<73}║")

print("║                                                                          ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")

print("\n\nArchivos generados:")
print("  - data/processed/regime_labels.parquet")
print("  - configs/regime_cluster_config.yaml")

print("\nSiguiente paso: 05_baselines_mpt.ipynb")